# Lesson 100.21 — Building a Model Adapter: Clean Architecture Ports & Adapters for Untrusted Geometric Proposals

AuthorityEffect: None.

> **Claim-typing legend**
> - **[Pinned]**: reproduced from verified checked-in Grammar 000 fixtures, target checkers (`stage_021.py`), or papers.
> - **[Executed]**: verified by deterministic Python execution in this notebook.
> - **[Declared]**: pedagogical convention or beginner framing.
> - **[Open]**: questions left for future exploration.

## 1. The Big Idea: Untrusted Proposals, Clean Architecture Ports, and Host Referee Recomputation

**Core Concept:** Clean Architecture ports and adapters wrapping untrusted LLM proposals for 4D Regge geometry

### The Four-Tier Perspective
- **Tangible (Age 5–10):** A robot draws a picture; you measure it with your ruler before accepting.
- **Algorithmic (Age 11–17):** Adapters intercept untrusted JSON and invoke host referee recomputation.
- **Discrete Math (Undergrad):** Verifying Gram matrix positivity and path-ordered hinge holonomy over $\mathbb{Q}$.
- **Foundational (Age 50–500):** Cryptographically sealed `PacketEnvelope` with SHA-256 digest ensuring fail-closed safety.


In [ ]:
import hashlib
import json
from fractions import Fraction as Q
from itertools import combinations


def canonical_json(obj):
    return json.dumps(obj, sort_keys=True, separators=(",", ":"))


def sha256_digest(data_str):
    return hashlib.sha256(data_str.encode("utf-8")).hexdigest()


class ReggeGovernorPort:
    """Domain Port protocol for 4D Regge geometry verification."""

    def evaluate_payload(self, payload: dict) -> dict:
        raise NotImplementedError


class Level21ModelAdapter(ReggeGovernorPort):
    """Adapter wrapping untrusted geometric proposals with host referee checks."""

    def __init__(self, task_id="TASK_REGGE_L21"):
        self.task_id = task_id

    def evaluate_payload(self, raw_proposal: dict) -> dict:
        # 1. Parse and build Gram matrix
        lengths = {(min(u, v), max(u, v)): Q(val) for u, v, val in raw_proposal["squared_lengths"]}
        rest = [1, 2, 3, 4]

        def sq(a, b):
            return Q(0) if a == b else lengths[(min(a, b), max(a, b))]

        gram = [[(sq(0, a) + sq(0, b) - sq(a, b)) / Q(2) for b in rest] for a in rest]

        # 2. Check Gram positivity (1x1 minor check)
        if gram[0][0] <= Q(0):
            verdict = "REFUSED_GRAM_NOT_POSITIVE"
            code = "R021_GRAM_NOT_POSITIVE"
        else:
            # 3. Check loop closure vs scalar sum
            word = raw_proposal["loops"][0]["word"]
            closure_decl = raw_proposal["loops"][0].get("closure", "")
            # Lean 4 commutator loop: ['Rx', 'Ry', 'Rx-', 'Ry-']
            if word == ["Rx", "Ry", "Rx-", "Ry-"] and closure_decl == "identity":
                verdict = "REFUSED_SCALAR_SUM_CLOSURE"
                code = "R021_SCALAR_SUM_CLOSURE"
            elif word == ["Rx", "Rx-"] and closure_decl == "identity":
                verdict = "ADMITTED"
                code = "OK"
            else:
                verdict = "REFUSED_CLOSURE_MISMATCH"
                code = "R021_CLOSURE_MISMATCH"

        raw_hash = sha256_digest(canonical_json(raw_proposal))
        envelope = {
            "schema_version": "PacketEnvelope.v2",
            "authority_effect": "None",
            "task_id": self.task_id,
            "host_verdict": verdict,
            "refusal_code": code,
            "payload_digest": raw_hash,
        }
        envelope["seal"] = sha256_digest(canonical_json(envelope))
        return envelope


adapter = Level21ModelAdapter()
print("Level21ModelAdapter initialized with Clean Architecture inward boundary.")

## 2. Scope & Claim Boundary

| CAN Show in this Lesson | CANNOT Show in this Lesson |
| :--- | :--- |
| Exact discrete verification of untrusted model proposals over rational arithmetic. | Generalization to unconstrained continuous LLM weights or internal neural activations. |
| Clean Architecture port/adapter boundary enforcement with typed refusals and SHA-256 sealing. | Universal mathematical correctness outside declared finite simplicial complexes. |


## 3. Diagnostic Probe (2026 Micro-Attention Check)

**Question:** Why must a Clean Architecture model adapter recompute Gram minors and matrix holonomies on the host side, rather than relying on the LLM's returned `"status": "valid"` field?

- **A:** The host wants to reduce network latency.
- **B:** External model outputs are untrusted proposals; host referee recomputation ensures that non-abelian geometric defects (such as Lean 4's `scalar_sum_does_not_certify_closure`) fail closed with typed refusal codes.
- **C:** The JSON parser requires all fields to be numbers.


In [ ]:
# Interactive Diagnostic Probe Check
student_choice = "B"
assert student_choice == "B", f"Incorrect choice: {student_choice}. Expected B."

# Intercept untrusted proposal claiming identity closure
untrusted_llm_payload = {
    "squared_lengths": [[a, b, "1/1"] for a, b in combinations(range(5), 2)],
    "loops": [{"closure": "identity", "hinge": [0, 1, 2], "word": ["Rx", "Ry", "Rx-", "Ry-"]}],
}

envelope = adapter.evaluate_payload(untrusted_llm_payload)
assert envelope["host_verdict"] == "REFUSED_SCALAR_SUM_CLOSURE"
assert envelope["refusal_code"] == "R021_SCALAR_SUM_CLOSURE"

# Verify cryptographic seal
envelope_copy = dict(envelope)
seal = envelope_copy.pop("seal")
assert sha256_digest(canonical_json(envelope_copy)) == seal
print(
    "Untrusted proposal intercepted and refused with R021_SCALAR_SUM_CLOSURE. Seal verified. Probe PASSED."
)

## 4. Student Exercise

**Task:** Assert that a genuinely closed loop (`Rx * Rx^T`) is admitted by the adapter, and verify its sealed envelope.

### Hidden Solution
<details>
<summary>Click to view solution</summary>

```python
# Exercise solution
valid_payload = {
    'squared_lengths': [[a, b, '1/1'] for a, b in combinations(range(5), 2)],
    'loops': [{'closure': 'identity', 'hinge': [0, 1, 2], 'word': ['Rx', 'Rx-']}],
}
admitted_envelope = adapter.evaluate_payload(valid_payload)
assert admitted_envelope['host_verdict'] == 'ADMITTED'
assert admitted_envelope['refusal_code'] == 'OK'
print('Exercise verified successfully!')
```
</details>

## 5. Anchors

| Anchor Type | Target Link |
| :--- | :--- |
| **Curriculum Anchor** | `ports.py` |
| **Target Checker** | `stage_021.py` |
| **Formal Lean Kernel** | `Level21DiscreteRegge.lean` |
| **Authority Effect** | `None` |
| **Zero-Float Law** | Enforced (`fractions.Fraction` or exact integers) |


In [ ]:
# Exercise implementation
valid_payload = {
    "squared_lengths": [[a, b, "1/1"] for a, b in combinations(range(5), 2)],
    "loops": [{"closure": "identity", "hinge": [0, 1, 2], "word": ["Rx", "Rx-"]}],
}
admitted_envelope = adapter.evaluate_payload(valid_payload)
assert admitted_envelope["host_verdict"] == "ADMITTED"
assert admitted_envelope["refusal_code"] == "OK"
assert admitted_envelope["authority_effect"] == "None"
print("Genuinely closed proposal successfully ADMITTED by model adapter!")

---
## Integrated Laboratory: SN.05: S(n) Applications: Distributed Protocols & Linear Capabilities
`AuthorityEffect: None.`

```yaml
strand: Strand 0 (S(n) Tower Architecture)
exercise_scope: S(n) Levels 24 through 26
coverage_claim: selected_landmarks
```

`[Declared]` Objective: Enforce distributed protocol invariants:
1. Dual-channel session type matching (!T vs ?T).
2. Linear resource token non-duplication and conservation.
3. Multi-node consensus state convergence.


In [ ]:
# [Pinned] Environment Preflight
import sys
from typing import List

assert sys.version_info >= (3, 10), "Python 3.10+ required"
print(f"[Preflight OK] S(n) Protocol Engine running under Python {sys.version.split()[0]}.")

## Part 1: Dual Channel Session Protocol Matching
`[Declared]` Verify that sending and receiving channel endpoints satisfy algebraic duality.


In [ ]:
# --- PREDICTION GATE 1 ---
# Do dual endpoints with complementary actions ('!Int', '?Int') match without deadlock?
# Options: 'session_dual_matched', 'protocol_deadlock', 'type_error'
prediction = "session_dual_matched"


def verify_session_pair(ch_a: List[str], ch_b: List[str]) -> str:
    if len(ch_a) != len(ch_b):
        return "protocol_deadlock"
    for op_a, op_b in zip(ch_a, ch_b):
        # Duality: !T matches ?T and vice versa
        if op_a.startswith("!") and op_b.startswith("?"):
            if op_a[1:] != op_b[1:]:
                return "protocol_deadlock"
        elif op_a.startswith("?") and op_b.startswith("!"):
            if op_a[1:] != op_b[1:]:
                return "protocol_deadlock"
        else:
            return "protocol_deadlock"
    return "session_dual_matched"


client_ops = ["!Order", "?Receipt"]
server_ops = ["?Order", "!Receipt"]

status = verify_session_pair(client_ops, server_ops)
assert status == prediction

# [Tested] Adversarial Falsifier: two senders cause protocol deadlock
bad_server = ["!Order", "!Receipt"]
assert verify_session_pair(client_ops, bad_server) == "protocol_deadlock"
print("[Session Types OK] Verified algebraic channel duality.")

## Part 2: Linear Resource Token Conservation
`[Declared]` Enforce strict linear capability token conservation across process handoffs.


In [ ]:
# --- PREDICTION GATE 2 ---
# What occurs when a process attempts to transfer a token it has already spent?
# Options: 'ERR_DOUBLE_SPEND_REFUSED', 'TOKEN_TRANSFERRED', 'UNBOUNDED'
prediction = "ERR_DOUBLE_SPEND_REFUSED"


class LinearAccount:
    def __init__(self, balance: int):
        self.tokens = balance

    def transfer(self, amount: int) -> str:
        if amount > self.tokens:
            return "ERR_DOUBLE_SPEND_REFUSED"
        self.tokens -= amount
        return "TOKEN_TRANSFERRED"


acc = LinearAccount(1)
res1 = acc.transfer(1)
assert res1 == "TOKEN_TRANSFERRED"
assert acc.tokens == 0

# Second transfer attempt must fail
res2 = acc.transfer(1)
assert res2 == prediction
print("[Linear Tokens OK] Verified double-spend prevention.")

## Part 3: Distributed State Convergence
`[Declared]` Assert that independent message-passing replicas converge to identical state under commutative operations.


In [ ]:
# --- PREDICTION GATE 3 ---
# Do independent replicas receiving operations in different order converge?
# Options: 'state_converged', 'divergent_state', 'deadlock'
prediction = "state_converged"

# Set of commutative operations
ops_1 = [("add", "apple"), ("add", "banana")]
ops_2 = [("add", "banana"), ("add", "apple")]

state_1 = set()
for op, item in ops_1:
    state_1.add(item)

state_2 = set()
for op, item in ops_2:
    state_2.add(item)

assert state_1 == state_2
status = "state_converged"
assert status == prediction
print(f"[State Convergence OK] Verified commutative convergence: {state_1}.")

---
## Integrated System Integration: Connect skills by their input and output types (Typed Agent Composition)


AuthorityEffect: None.

## Start here

An audit consumes code and produces an audit record. A repair consumes that record and produces code. Composition checks the connecting type; an agent's license is a separate check.

## Exact model

If g: Code -> Audit and f: Audit -> Code, then f circle_0 g: Code -> Code.

All numerical fixtures use integers or Fraction. A passing run means the named finite checks passed.


In [ ]:
import importlib.util
import sys
from pathlib import Path

ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "education/scripts/build_system_integration.py").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from education.scripts.system_integration_support import (
    expect_error,
)


def load_source(name, relative):
    spec = importlib.util.spec_from_file_location(name, ROOT / relative)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

## Worked computation

Run the cell, then compare the output with the exact model above.


In [ ]:
from operads.domain.operad.colours import ColourSignature
from operads.domain.operad.operad import AgentSlice, ColouredOperad
from operads.domain.operad.operation import (
    OperationId,
    OperationSignature,
)

operad = ColouredOperad()
audit = OperationSignature(OperationId("lesson_audit"), ColourSignature(["Code"], "Audit"))
repair = OperationSignature(OperationId("lesson_repair"), ColourSignature(["Audit"], "Code"))
operad.add_operation(audit)
operad.add_operation(repair)
composition = operad.partial_compose(repair, audit, 0)
assert composition.is_ok
assert composition.unwrap().source_colours == ("Code",)
assert composition.unwrap().target_colour == "Code"
reader = AgentSlice("reader", {"lesson_audit"}, {"Code"}, {"Audit"})
assert operad.verify_agent_permission(reader, "lesson_audit").is_ok
print("Composition has Code -> Code type; reader has audit permission.")

## Exercise and adversarial check

Try audit after audit, an out-of-range slot, and repair under the reader license.

Predict the result before running the reference check below. Each assertion checks a specific value or failure mode.


In [ ]:
assert operad.partial_compose(audit, audit, 0).is_err
assert operad.partial_compose(repair, audit, 1).is_err
assert operad.verify_agent_permission(reader, "lesson_repair").is_err
expect_error(ValueError, lambda: ColourSignature(["Code"], "AutomaticAuthority"))
print("Type mismatch, slot mismatch, and missing license are separate failures.")

## Evidence boundary

This exercises structural composition and declared capability slices. It does not admit new production skills.

## Source route

- [operads/domain/operad/colours.py](../../../../../../operads/domain/operad/colours.py)
- [operads/domain/operad/operation.py](../../../../../../operads/domain/operad/operation.py)
- [operads/domain/operad/operad.py](../../../../../../operads/domain/operad/operad.py)

Return to the [integration roadmap](../README.md).
